## Lesson 07 Homework Exercises

In [14]:
import os
import re
from scipy import stats
import gzip
from Bio import SeqIO
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
DIR = '/Users/fernandojuarez/Documents/GitHub/Python_Study/QBRP/'
DIR_ext = '/Users/fernandojuarez/Documents/01 - Learning/QBRP/'
# 'Lesson 07/Pfam-A.regions.uniprot.tsv.gz'

# Exercise 1

## Part A
Download from Pfam’s FTP site all the occurrences of Pfam domains/HMM profiles in the entire human
proteome. Use version 31.0 (not the newer ones). It should be a compressed .tsv file with ~100k rows
and 14 columns. Each row in this file indicates an occurrence of a single domain/HMM profile in a single
protein (each profile can occur multiple times in different proteins, and even in the same protein, and
each protein can have multiple profiles).

Among the fields in this file are:

● seq id: the UniProt ID of the protein where the profile occurs.

● alignment start & alignment end: 1-based coordinates indicating the position of the profile within the protein.

● hmm acc & hmm name: The accession ID and the name of the profile occurring in the protein.

● type: the type of the occurring HMM profile (most commonly Domain).

How many occurrences are there of each type?

In [3]:
df = pd.read_csv(os.path.join(DIR, '07_Lesson/Files/9606.tsv.gz'), skiprows=2, sep = r'\t|> <', \
                 engine='python', na_values= ['No_clan'])\
                    .rename(columns= lambda name: re.sub(r'[ \-]', '_', re.sub(r'[#\<\>]', '',name)))

In [5]:
display(df.head())

,seq_id,alignment_start,alignment_end,envelope_start,envelope_end,hmm_acc,hmm_name,type,hmm_start,hmm_end,hmm_length,bit_score,E_value,clan
0,A0A024QZ18,69,147,66,147,PF00595,PDZ,Domain,4,82,82,51.3,1.600000e-10,CL0466
1,A0A024QZ33,5,123,4,123,PF09745,DUF2040,Coiled-coil,2,121,121,124.9,2.600000e-33,NaN
2,A0A024QZ42,25,84,22,86,PF13499,EF-hand_7,Domain,4,69,71,41.7,1.800000e-07,CL0220
3,A0A024QZB8,40,436,39,437,PF02487,CLN3,Family,2,398,399,461.1,4.800000e-135,NaN
4,A0A024QZP7,4,287,4,287,PF00069,Pkinase,Domain,1,264,264,258.8,7.400000e-74,CL0016


In [4]:
display(df['type'].value_counts())

type
Domain         57203
Family         31138
Repeat          8354
Motif            761
Coiled-coil      508
Disordered       272
Name: count, dtype: int64

## Part B
Filter only the records of type Domain. What is the average length of Pfam domains? What are the 5
longest and 5 shortest domains (on average)?

Note that the field hmm_length indicates the theoretical length of the Pfam profile, not the actual
length of its occurrence within a given protein.

In [5]:
display(df[df['type'] == 'Domain'])

,seq_id,alignment_start,alignment_end,envelope_start,envelope_end,hmm_acc,hmm_name,type,hmm_start,hmm_end,hmm_length,bit_score,E_value,clan
0,A0A024QZ18,69,147,66,147,PF00595,PDZ,Domain,4,82,82,51.3,1.600000e-10,CL0466
2,A0A024QZ42,25,84,22,86,PF13499,EF-hand_7,Domain,4,69,71,41.7,1.800000e-07,CL0220
4,A0A024QZP7,4,287,4,287,PF00069,Pkinase,Domain,1,264,264,258.8,7.400000e-74,CL0016
5,A0A024QZX5,11,380,10,380,PF00079,Serpin,Domain,2,370,370,435.8,2.300000e-127,NaN
6,A0A024R0K5,40,140,38,141,PF07686,V-set,Domain,3,108,109,47.4,2.600000e-09,CL0011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98216,X6RKW4,219,311,216,312,PF00254,FKBP_C,Domain,5,93,94,73.7,1.700000e-17,CL0487
98218,X6RL08,10,87,9,87,PF13873,Myb_DNA-bind_5,Domain,2,78,78,83.6,1.100000e-20,CL0123
98220,X6RLA8,48,199,48,199,PF07177,Neuralized,Domain,1,149,150,137.2,6.300000e-37,CL0004
98221,X6RLJ0,116,219,116,220,PF00386,C1q,Domain,1,105,127,96.6,1.800000e-24,CL0100


In [6]:
display(df[df['type'] == 'Domain']['hmm_name'].value_counts())

hmm_name
zf-C2H2          8087
fn3              1696
I-set            1259
Cadherin         1216
Pkinase           996
                 ... 
MVP_shoulder        1
BH3                 1
DNMT1-RFD           1
HTH_40              1
Bclx_interact       1
Name: count, Length: 2309, dtype: int64

In [ ]:
# df[df['type'] == 'Domain']['hmm_name'].value_counts().to_clipboard()

In [7]:
df['alignment_length'] = df['alignment_end'] - df['alignment_start'] + 1
display(df[df['type'] == 'Domain']['alignment_length'].mean())

np.float64(88.59769592503889)

In [8]:
display(df[df['type'] == 'Domain'].nlargest(5, 'alignment_length'))
display(df[df['type'] == 'Domain'].nsmallest(5, 'alignment_length'))

,seq_id,alignment_start,alignment_end,envelope_start,envelope_end,hmm_acc,hmm_name,type,hmm_start,hmm_end,hmm_length,bit_score,E_value,clan,alignment_length
68345,Q14683,3,1208,3,1211,PF02463,SMC_N,Domain,1,216,220,164.9,2.700000e-45,CL0023,1206
81274,Q8NDV3,4,1203,3,1207,PF02463,SMC_N,Domain,2,215,220,206.2,6.000000e-58,CL0023,1200
95139,Q9UQE7,2,1196,2,1197,PF02463,SMC_N,Domain,1,219,220,214.8,1.400000e-60,CL0023,1195
27115,E9PD53,58,1245,58,1249,PF02463,SMC_N,Domain,1,216,220,206.8,4.100000e-58,CL0023,1188
91727,Q9NTJ3,83,1270,83,1274,PF02463,SMC_N,Domain,1,216,220,206.7,4.300000e-58,CL0023,1188


,seq_id,alignment_start,alignment_end,envelope_start,envelope_end,hmm_acc,hmm_name,type,hmm_start,hmm_end,hmm_length,bit_score,E_value,clan,alignment_length
16462,B1AKI6,62,72,59,72,PF14608,zf-CCCH_2,Domain,8,18,18,10.3,1200.0,CL0537,11
30797,F5H0A9,304,314,303,314,PF13912,zf-C2H2_6,Domain,2,12,27,13.1,110.0,CL0361,11
47994,K7EK80,299,309,298,309,PF13912,zf-C2H2_6,Domain,2,12,27,12.8,130.0,CL0361,11
48163,K7EKZ8,174,184,173,184,PF13912,zf-C2H2_6,Domain,2,12,27,14.9,30.0,CL0361,11
54967,O95205,61,71,58,71,PF14608,zf-CCCH_2,Domain,8,18,18,8.8,3500.0,CL0537,11


## Part C
Compare the lengths of the G-alpha and tyrosine kinase domains. Is there a large difference in their
average lengths?

Is this difference significant? Consider using Scipy’s t-test [https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html]

(see this video [https://www.youtube.com/watch?v=0Pd3dc1GcHc] for a refresher on t-tests). Other
appropriate statistical tests will also be accepted.

In [9]:
display(df[df['hmm_name'] == 'G-alpha']['alignment_length'])
display(df[df['hmm_name'] == 'Pkinase_Tyr']['alignment_length'])

3570      99
5455     123
14000     51
15631    330
16597    152
20147    130
21132    141
26947    194
37038    149
41050    117
41144     82
42119     79
48208    196
48848    179
49024    128
55407    329
56299    331
56977    329
57067    329
57898    330
59239    331
59240    330
60779    330
60971    341
61700    348
62788    329
64594    363
64595    329
65760    333
68014    339
71784     51
71785    157
71786    364
97202    135
Name: alignment_length, dtype: int64

654      281
1321     266
2195     117
3245      57
3585     254
        ... 
96928    112
97161    277
97572    167
97597    257
97858    250
Name: alignment_length, Length: 299, dtype: int64

In [10]:
t_statistic, p_value = stats.ttest_ind(df[df['hmm_name'] == 'G-alpha']['alignment_length'],\
                                        df[df['hmm_name'] == 'Pkinase_Tyr']['alignment_length'])

print(t_statistic, p_value)

# There is a better way to display the values using that f notation

0.7436218990299451 0.4576327609911348


## Part D
Download the required protein sequences from UniProt and add a new column to the table for the
entire protein sequences of the records. Add another column for the sequence of the HMM profile
within the protein (i.e. the latter is a subsequence of the former).

In [ ]:
file_path = '07_Lesson/Files/uniprotkb_taxonomy_id_9606_2026_01_07.fasta.gz'

# prepare dict for record storage
uniprot_id_seq = {}

with gzip.open(os.path.join(DIR, file_path), 'rt') as handle:
    for record in SeqIO.parse(handle, 'fasta'):
        # print(f'ID: {record.id}')
        # print(f'Sequence: {record.seq}')
        # print(record.id.split('|')[1])
        # print(record)
        # print('~~~~~~~~~~~~~~~~')
        uniprot_id_seq[record.id.split('|')[1]] = str(record.seq)


In [ ]:
uniprot_id_seq

{'A0A024QYR6': 'MERGGEAAAAAAAAAAAPGRGSESPVTISRAGNAGELVSPLLLPPTRRRRRRHIQGPGPVLNLPSAAAAPPVARAPEAAGGGSRSEDYSSSPHSAAAAARPLAAEEKQAQSLQPSSSRRSSHYPAAVQSQAAAERGASATAKSRAISILQKKPRHQQLLPSLSSFFFSHRLPDMTAIIKEIVSRNKRRYQEDGFDLDLTYIYPNIIAMGFPAERLEGVYRNNIDDVVRFLDSKHKNHYKIYNLCAERHYDTAKFNCRVAQYPFEDHNPPQLELIKPFCEDLDQWLSEDDNHVAAIHCKAGKGRTGVMICAYLLHRGKFLKAQEALDFYGEVRTRDKKGVTIPSQRRYVYYYSYLLKNHLDYRPVALLFHKMMFETIPMFSGGTCNPQFVVCQLKVKIYSSNSGPTRREDKFMYFEFPQPLPVCGDIKVEFFHKQNKMLKKDKMFHFWVNTFFIPGPEETSEKVENGSLCDQEIDSICSIERADNDKEYLVLTLTKNDLDKANKDKANRYFSPNFKVKLYFTKTVEEPSNPEASSSTSVTPDVSDNEPDHYRYSDTTDSDPENEPFDEDQHTQITKV',
 'A0A024R1X5': 'MEGSKTSNNSTMQVSFVCQRCSQPLKLDTSFKILDRVTIQELTAPLLTTAQAKPGETQEEETNSGEEPFIETPRQDGVSRRFIPPARMMSTESANSFTLIGEASDGGTMENLSRRLKVTGDLFDIMSGQTDVDHPLCEECTDTLLDQLDTQLNVTENECQNYKRCLEILEQMNEDDSEQLQMELKELALEEERLIQELEDVEKNRKIVAENLEKVQAEAERLDQEEAQYQREYSEFKRQQLELDDELKSVENQMRYAQTQLDKLKKTNVFNATFHIWHSGQFGTINNFRLGRLPSVPVEWNEINAAWGQTVLLLHALANKMGLKFQRYRLVPYGNHSYLESLTDKSKELPLYCSGGLRFFWDNKFDHAMVAFLDCVQQFKEEVEKGETR

## Part E
What is the biological role of the PDZ domain?

Save the sequences of all the occurrences of the PDZ domain as a FASTA file. Submit it to an online
https://www.ebi.ac.uk/Tools/msa/clustalo/) and create a logo
Multiple Sequence Alignment tool (e.g. from it (e.g. using http://www.cbs.dtu.dk/biotools/Seq2Logo-2.1/).

Compare your logo to Pfam’s HMM logo for this domain. Identify similarities and differences between
the two logos, and suggest plausible explanations for the differences.

## Part F
There’s one protein with 9 occurrences of the PDZ domain. Find it.

What is known about the function of this protein?

## Part G [Optional, recommended]

What percentage of the exon junctions in the coding region of the transcript of the protein you have
found occur within a PDZ domain? What percentage would you expect to see at random?

## Part H [Optional, recommended]
At which locations inside the PDZ domain do exon junctions occur (across the entire human proteome)?
How are they distributed compared to random uniform distribution?

## Part I [Optional, recommended]
Do PDZ occurrences within the same protein more similar to each other than occurrences in different
proteins?

# Bonus Question 1

What is “time series” data and why is it important in biological and medical research?

Demonstrate Panda’s time-series capabilities with an example biological/medical dataset (e.g. you may
use the BioTIME dataset [http://biotime.st-andrews.ac.uk/home.php]).